In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 15. Multi-Head Attention — テキスト テキスト テキスト

> **学習目標**
> - Multi-Head Attentionテキスト テキスト テキスト テキスト テキスト テキスト テキスト
> - テキスト テキスト $h$テキスト テキスト 次元 $d_k = d/h$テキスト テキスト テキスト
> - MHA, MQA, GQAテキスト テキスト テキスト·速度 テキスト 比較テキスト

## 15.1 テキスト テキスト テキスト

テキスト Attention テキスト テキスト テキスト テキスト(テキスト: テキスト-テキスト)テキスト 学習テキスト テキスト. **テキスト テキスト テキスト 学習**テキスト?

## 15.2 Multi-Head Attention (MHA)

$$\mathrm{MultiHead}(Q, K, V) = \mathrm{Concat}(\mathrm{head}_1, \ldots, \mathrm{head}_h) W^O$$
$$\mathrm{head}_i = \mathrm{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

- $W_i^Q \in \mathbb{R}^{d \times d_k}$, $W_i^K \in \mathbb{R}^{d \times d_k}$, $W_i^V \in \mathbb{R}^{d \times d_v}$
- $W^O \in \mathbb{R}^{h d_v \times d}$
- テキスト $d_k = d_v = d/h$

テキスト: $d = 512, h = 8$ → $d_k = 64$. テキスト テキスト 64次元テキスト Attention.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        # Q, K, V, O テキスト (テキスト テキスト)
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x, mask=None):
        B, T, D = x.shape
        # (B, T, 3D) -> 3テキスト (B, T, D)
        qkv = self.W_qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        # テキスト 次元 テキスト: (B, T, D) -> (B, T, H, d_k) -> (B, H, T, d_k)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        # Attention: (B, H, T, d_k) @ (B, H, d_k, T) -> (B, H, T, T)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        # (B, H, T, d_k) -> (B, T, H, d_k) -> (B, T, D)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out), weights

# テキスト
d_model, n_heads = 64, 8
mha = MultiHeadAttention(d_model, n_heads)
x = torch.randn(2, 10, d_model)
out, weights = mha(x)
print(f"Input: {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention Weight: {weights.shape} (B, H, T, T)")
print(f"Parameter Count: {sum(p.numel() for p in mha.parameters()):,}")


## 15.3 テキスト テキスト テキスト

テキスト テキスト テキスト テキスト テキスト 学習テキスト テキスト:
- テキスト テキスト: テキスト テキスト テキスト (positional head)
- テキスト テキスト: テキスト テキスト (テキスト-テキスト)
- テキスト テキスト: テキスト テキスト


In [ ]:
# テキスト Attention テキスト 可視化
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, n_heads=4)

# テキスト テキスト (4 テキスト)
x = torch.randn(1, 8, 32)
out, weights = mha(x)

# テキスト テキスト Attention テキスト
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
tokens = [f'tok{i}' for i in range(8)]
for h, ax in enumerate(axes):
    im = ax.imshow(weights[0, h].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {h}')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('../figures/ch15_heads_patterns.png', dpi=100, bbox_inches='tight')
plt.show()
print("テキスト Headテキスト テキスト Attention Patternテキスト Trainingテキスト (テキスト テキスト Training).")


## 15.4 Multi-Query Attention (MQA)

**問題**: MHAテキスト KV テキスト $O(L \cdot h \cdot d_k)$ → 推論 テキスト テキスト テキスト.

**MQA** (Shazeer, 2019): $Q$テキスト テキスト, $K, V$テキスト テキスト テキスト テキスト.

$$\mathrm{head}_i = \mathrm{Attention}(Q W_i^Q, K W^K, V W^V)$$

- テキスト テキスト, KV テキスト 1/hテキスト テキスト
- テキスト テキスト テキスト 推論 速度 テキスト テキスト

## 15.5 Grouped-Query Attention (GQA)

**GQA** (Ainslie et al., 2023): テキスト テキスト.
- $h$テキスト Q テキスト, $g$テキスト KV テキスト テキスト ($1 \leq g \leq h$)
- $g=1$: MQA, $g=h$: MHA
- **LLaMA-2テキスト テキスト** (テキスト $g = h/8$)


In [ ]:
# MHA, MQA, GQA テキスト 比較
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_q_heads == 0
        self.d_model = d_model
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.d_k = d_model // n_q_heads
        # Q: n_q_heads * d_k, K/V: n_kv_heads * d_k
        self.W_q = nn.Linear(d_model, n_q_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(n_q_heads * self.d_k, d_model, bias=False)
        # テキスト Q テキスト テキスト
        self.n_rep = n_q_heads // n_kv_heads

    def forward(self, x, mask=None):
        B, T, D = x.shape
        q = self.W_q(x).view(B, T, self.n_q_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        # KVテキスト Q テキスト テキスト テキスト (group expansion)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_o(out), weights

# 比較: MHA, GQA, MQA
d = 64
configs = [
    ('MHA (h=8)', 8, 8),
    ('GQA (g=4)', 8, 4),
    ('GQA (g=2)', 8, 2),
    ('MQA (g=1)', 8, 1),
]
print(f"{'Config':>12} | {'Params':>10} | {'KV size':>10}")
print("-" * 38)
for name, nq, nkv in configs:
    attn = GroupedQueryAttention(d, nq, nkv)
    n_params = sum(p.numel() for p in attn.parameters())
    kv_size = nkv * (d // nq)  # KV Headテキスト Dimension
    print(f"{name:>12} | {n_params:>10,} | {kv_size:>10}")


## 15.6 [CPU/GPU ベンチマーク ⑥] MHA vs MQA vs GQA

推論 テキスト KV テキスト テキスト 速度 テキスト 比較.


In [ ]:
# MHA/GQA/MQA 推論 ベンチマーク
from llm_math.bench import time_fn

configs = [
    ('MHA', 8, 8),
    ('GQA-4', 8, 4),
    ('GQA-2', 8, 2),
    ('MQA', 8, 1),
]
B, T, D = 1, 256, 256
x = torch.randn(B, T, D)

print(f"\n{'Config':>8} | {'Params':>8} | {'CPU (ms)':>12} | {'GPU (ms)':>12}")
print("-" * 50)
for name, nq, nkv in configs:
    attn = GroupedQueryAttention(D, nq, nkv)
    n_params = sum(p.numel() for p in attn.parameters())
    res = time_fn(lambda x: attn(x), x, device='cpu', warmup=2, repeat=3)
    if torch.cuda.is_available():
        attn_gpu = GroupedQueryAttention(D, nq, nkv).cuda()
        x_gpu = x.cuda()
        res_gpu = time_fn(lambda x: attn_gpu(x), x_gpu, device='cuda', warmup=2, repeat=5)
        print(f"{name:>8} | {n_params:>8,} | {res['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f}")
    else:
        print(f"{name:>8} | {n_params:>8,} | {res['mean_ms']:>12.3f} | {'N/A':>12}")


## 15.7 要点

| テキスト | Q テキスト | KV テキスト | KV テキスト | テキスト |
|---|---|---|---|---|
| MHA | $h$ | $h$ | $O(Lhd_k)$ | テキスト テキスト |
| MQA | $h$ | 1 | $O(Ld_k)$ | 速度 テキスト |
| GQA | $h$ | $g$ | $O(Lgd_k)$ | LLaMA-2 テキスト |

## 演習問題

1. MHAテキスト テキスト テキスト 1, 4, 8, 16テキスト テキスト 学習 テキスト 比較テキスト.
2. Attention テキスト テキスト 可視化テキスト, "positional head" (テキスト テキスト)テキスト テキスト テキスト.
3. MQAテキスト MHAテキスト テキスト テキスト テキスト 計算テキスト, テキスト 検証テキスト.
4. GQAテキスト $g = 1, 2, 4, 8$テキスト テキスト 推論 速度テキスト テキスト 比較テキスト.
5. Multi-Head Attentionテキスト $W^O$ 行列テキスト テキスト テキスト テキスト.

> 解答: `solutions/ch15_solutions.ipynb`
